In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from pybaseball import statcast

def load_statcast_range(start_date, end_date, step_days=7):
    start = datetime.strptime(start_date, "%Y-%m-%d")
    end   = datetime.strptime(end_date, "%Y-%m-%d")

    all_dfs = []
    current = start

    while current <= end:
        chunk_start = current.strftime("%Y-%m-%d")
        chunk_end = min(current + timedelta(days=step_days), end).strftime("%Y-%m-%d")
        print(f"Downloading: {chunk_start} → {chunk_end}")

        try:
            chunk = statcast(start_dt=chunk_start, end_dt=chunk_end)
            if chunk is not None and not chunk.empty:
                all_dfs.append(chunk)
        except Exception as e:
            print(f"Failed on {chunk_start} → {chunk_end}: {e}")

        current += timedelta(days=step_days + 1)

    if len(all_dfs) == 0:
        raise ValueError("No data downloaded — check date ranges.")
    return pd.concat(all_dfs, ignore_index=True)

df = load_statcast_range("2025-03-28", "2025-10-01", step_days=30)
df = df[df["game_type"] == "R"]

Downloading: 2025-03-28 → 2025-04-27
This is a large query, it may take a moment to complete


100%|██████████████████████████████████████████████████████████████████████████████████| 31/31 [00:24<00:00,  1.28it/s]


Downloading: 2025-04-28 → 2025-05-28
This is a large query, it may take a moment to complete


100%|██████████████████████████████████████████████████████████████████████████████████| 31/31 [00:21<00:00,  1.46it/s]


Downloading: 2025-05-29 → 2025-06-28
This is a large query, it may take a moment to complete


...

100%|██████████████████████████████████████████████████████████████████████████████████| 31/31 [00:24<00:00,  1.27it/s]


Downloading: 2025-08-30 → 2025-09-29
This is a large query, it may take a moment to complete


100%|██████████████████████████████████████████████████████████████████████████████████| 31/31 [00:27<00:00,  1.13it/s]


Downloading: 2025-09-30 → 2025-10-01
This is a large query, it may take a moment to complete


100%|████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:05<00:00,  2.56s/it]


In [105]:
df = df.rename(columns={
    "estimated_woba_using_speedangle": "xwobacon"
})

SWING_EVENTS = {
    "foul",
    "foul_tip",
    "hit_into_play",
    "swinging_strike",
    "swinging_strike_blocked"
}

WHIFF_EVENTS = {
    "swinging_strike",
    "swinging_strike_blocked"
}

BUNT_EVENTS = {
    "bunt_foul_tip",
    "foul_bunt",
    "missed_bunt"
}

In [106]:
df["swing"] = (
    df["description"].isin(SWING_EVENTS)
    & ~df["description"].isin(BUNT_EVENTS)
).astype(int)

df["whiff"] = (
    df["description"].isin(WHIFF_EVENTS)
    & ~df["description"].isin(BUNT_EVENTS)
).astype(int)

df["in_play"] = df["description"] == "hit_into_play"

df["count"] = df["balls"].astype(str) + "-" + df["strikes"].astype(str)

In [107]:
df = df.sort_values(
    ["game_date", "player_name", "inning", "at_bat_number", "pitch_number"]
)

df["prev_pitch_type"] = df.groupby(["player_name", "game_date"])["pitch_type"].shift(1)

df = df.dropna(subset=["prev_pitch_type"])

In [108]:
group_cols = ["player_name", "pitch_type", "prev_pitch_type", "stand", "count"]

agg = df.groupby(group_cols).agg(
    pitches=("pitch_type", "count"),
    swings=("swing", "sum"),
    whiffs=("whiff", "sum"),
    balls_in_play=("in_play", "sum")
).reset_index()

contact_df = df[df["in_play"]]
contact_agg = contact_df.groupby(group_cols).agg(avg_xwobacon=("xwobacon", "mean")).reset_index()

agg = agg.merge(contact_agg, on=group_cols, how="left")

In [109]:
agg["whiff_rate"] = agg["whiffs"] / agg["swings"]
agg["whiff_rate"] = agg["whiff_rate"].fillna(0)

context_cols = ["player_name", "prev_pitch_type", "stand", "count"]

agg["context_max_xwoba"] = agg.groupby(context_cols)["avg_xwobacon"].transform("max")

agg["contact_success"] = agg["context_max_xwoba"] - agg["avg_xwobacon"]

agg["context_max_contact"] = agg.groupby(context_cols)["contact_success"].transform("max")

agg["contact_success"] = agg["contact_success"] / agg["context_max_contact"]
agg["contact_success"] = agg["contact_success"].fillna(0)

agg = agg.drop(columns=["context_max_xwoba", "context_max_contact"])

In [110]:
agg["success_score"] = (
    0.5 * agg["whiff_rate"] +
    0.5 * agg["contact_success"]
)

In [111]:
MIN_PITCHES = 15

agg = agg[agg["pitches"] >= MIN_PITCHES]

In [112]:
def rank_pitches(
    player_name,
    prev_pitch_type,
    stand,
    balls,
    strikes,
    data=agg
):
    count = f"{balls}-{strikes}"

    subset = data[
        (data["player_name"] == player_name) &
        (data["prev_pitch_type"] == prev_pitch_type) &
        (data["stand"] == stand) &
        (data["count"] == count)
    ]

    if subset.empty:
        return None

    ranked = subset.sort_values("success_score", ascending=False).copy()
    
    ranked = ranked[[
        "pitch_type", "success_score", "whiff_rate", "avg_xwobacon"
    ]].reset_index(drop=True)

    ranked["rank"] = ranked.index + 1

    return ranked

In [113]:
ranked_options = rank_pitches(
    player_name="Crochet, Garrett",
    prev_pitch_type="FF",
    stand="R",
    balls=1,
    strikes=2
)

print(ranked_options)

  pitch_type  success_score  whiff_rate  avg_xwobacon  rank
0         ST            0.7        0.40        0.2362     1
1         FF       0.476321        0.45        0.5375     2
